# 📐 Chapter 2: Bayes' Theorem

## *From a 1763 essay to the most powerful weapon at Bletchley Park*

---

### The Problem of Reasoning Under Uncertainty

In 1940, an Enigma intercept arrived. The codebreakers knew it was encrypted
with settings they didn't have. They knew, from intercepting traffic patterns,
that weather reports always began the same way — "WETTER" or similar.

They had a hypothesis ("maybe the first word is WETTER") and some evidence
(the first six letters of ciphertext). How confident should they be?
And — critically — how should that confidence *change* as they gathered
more evidence?

These questions had already been answered — by a Presbyterian minister
named **Thomas Bayes**, in a posthumous essay published in 1763.


## Part 1: Probability from First Principles

We need to establish some foundations before we can derive Bayes' theorem.

### Notation

- $P(A)$ — the probability that event $A$ occurs. Always between 0 and 1.
- $P(A \cap B)$ — the probability that **both** $A$ and $B$ occur.
- $P(A | B)$ — the probability of $A$ **given** that $B$ has already occurred.
  This is *conditional probability*.

### The Product Rule

How do we compute the probability that two things are both true?

$$P(A \cap B) = P(A \mid B) \cdot P(B)$$

In words: the probability that $A$ and $B$ both happen equals the probability
that $B$ happens, multiplied by the probability that $A$ happens *given* $B$
already happened.

But we can also write it the other way around:

$$P(A \cap B) = P(B \mid A) \cdot P(A)$$

These two expressions for the same quantity are the seed of Bayes' theorem.


In [ ]:
# Let's build intuition with a concrete simulation
import numpy as np
rng = np.random.default_rng(42)

# Scenario: We're scanning intercepted German messages.
# 30% of messages are weather reports (W).
# Given it's a weather report, 80% start with 'WETTER'.
# Given it's NOT a weather report, only 5% happen to start with 'WETTER'.

N = 100_000
is_weather   = rng.random(N) < 0.30
starts_wetter = np.where(
    is_weather,
    rng.random(N) < 0.80,   # P(WETTER | weather report)
    rng.random(N) < 0.05    # P(WETTER | other message)
)

# Empirical probabilities
p_weather  = is_weather.mean()
p_wetter   = starts_wetter.mean()
p_both     = (is_weather & starts_wetter).mean()
p_weather_given_wetter = p_both / p_wetter

print(f'P(weather report)       = {p_weather:.3f}')
print(f'P(starts WETTER)        = {p_wetter:.3f}')
print(f'P(both)                 = {p_both:.3f}')
print(f'P(weather | WETTER)     = {p_weather_given_wetter:.3f}')

## Part 2: Deriving Bayes' Theorem

From the product rule, we have two equal expressions for $P(H \cap E)$:

$$P(H \mid E) \cdot P(E) = P(E \mid H) \cdot P(H)$$

Divide both sides by $P(E)$:

$$\boxed{P(H \mid E) = \frac{P(E \mid H) \cdot P(H)}{P(E)}}$$

This is **Bayes' theorem**. Let's name each term:

| Term | Name | Meaning in our context |
|---|---|---|
| $P(H)$ | **Prior** | How likely we think this Enigma setting is *before* seeing any evidence |
| $P(E \mid H)$ | **Likelihood** | How probable the evidence is *if* this setting is correct |
| $P(H \mid E)$ | **Posterior** | Our updated belief *after* seeing the evidence |
| $P(E)$ | **Evidence** (normaliser) | Total probability of the evidence across all hypotheses |

The denominator $P(E)$ ensures the posterior is a proper probability:

$$P(E) = \sum_{i} P(E \mid H_i) \cdot P(H_i)$$

### The Bayesian Mantra

> **Posterior ∝ Likelihood × Prior**

The normalisation (dividing by $P(E)$) just makes everything sum to 1.
The real action is in the product on the right.


In [ ]:
# Verify our simulation matches the formula analytically

p_H   = 0.30   # P(weather report)  — the Prior
p_E_given_H     = 0.80  # P(WETTER | weather)  — the Likelihood (H is true)
p_E_given_not_H = 0.05  # P(WETTER | not weather) — Likelihood (H is false)

# P(E) = P(E|H)P(H) + P(E|¬H)P(¬H)  [law of total probability]
p_E = p_E_given_H * p_H + p_E_given_not_H * (1 - p_H)

# Bayes' theorem
p_H_given_E = (p_E_given_H * p_H) / p_E

print('Analytical Bayes\' theorem:')
print(f'  P(weather | WETTER) = {p_H_given_E:.4f}')
print()
print('Simulation result (from above):')
print(f'  P(weather | WETTER) = {p_weather_given_wetter:.4f}')
print()
print(f'Our prior was P(weather) = {p_H:.2f}.')
print(f'After seeing WETTER, our belief jumps to {p_H_given_E:.3f}.')
print('Evidence changed our mind significantly — that\'s Bayesian updating.')

## Part 3: Prior, Likelihood, Posterior — The Enigma Version

Now let's map this directly onto the Enigma problem.

**The hypotheses**: Each possible Enigma setting is a hypothesis $H_i$.
There are roughly $10^{23}$ of them — call the set $\{H_1, H_2, \ldots, H_N\}$.

**The prior** $P(H_i)$: Before we've seen any evidence, all settings are
equally plausible: $P(H_i) = 1/N$ for all $i$. This is the *uniform prior*.

**The evidence** $E$: We intercept a ciphertext. We have a *crib* —
a suspected piece of plaintext. For example, we suspect the message starts
with `WETTERVORHERSAGEBISKAYA` (weather forecast for the Bay of Biscay).

**The likelihood** $P(E \mid H_i)$: If setting $H_i$ were correct, how
probable is the observed ciphertext? This is where the Enigma constraint
becomes crucial:

- If, under setting $H_i$, the crib forces any letter to encrypt to itself
  → $P(E \mid H_i) = 0$. This setting is *impossible*.
- Otherwise, we score using letter frequencies.

**The posterior** $P(H_i \mid E)$: Updated probability for each setting.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.enigma_bayes.bayes.probability import uniform_prior, bayesian_update, normalize

# Toy example: only 5 possible settings
# Setting 3 is the correct one.
N_hypotheses = 5
prior = uniform_prior(N_hypotheses)
print('Prior:', prior)

# After one crib check:
# Settings 1, 2, 4 are eliminated by Enigma constraint (likelihood = 0)
# Setting 3 decrypts well (likelihood = 0.9)
# Setting 5 decrypts somewhat (likelihood = 0.1)
likelihoods_round1 = np.array([0.0, 0.0, 0.9, 0.0, 0.1])

posterior1 = bayesian_update(prior, likelihoods_round1)
print('After crib 1:', posterior1)

# After a second, independent crib check:
likelihoods_round2 = np.array([0.0, 0.0, 0.95, 0.0, 0.05])
posterior2 = bayesian_update(posterior1, likelihoods_round2)
print('After crib 2:', posterior2)

In [ ]:
# Visualise the updating process
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels = [f'H{i+1}' for i in range(N_hypotheses)]
colors = ['#4a90e2'] * N_hypotheses

for ax, probs, title in zip(axes,
    [prior, posterior1, posterior2],
    ['Prior (uniform)', 'Posterior after crib 1', 'Posterior after crib 2']):
    bars = ax.bar(labels, probs, color=colors, edgecolor='white', linewidth=0.8)
    bars[2].set_color('#c8a44a')  # highlight the correct setting (H3)
    ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Probability')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Bayesian Updating: Posterior Probability Collapses onto the Correct Setting',
             fontsize=12)
plt.tight_layout()
plt.savefig('../website/public/assets/bayes_update_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## Part 4: Why the Uniform Prior?

Some students ask: why start with a uniform prior? Isn't that just guessing?

Yes — and that's exactly right. The uniform prior is the mathematical way of
saying *"we have no information yet."* It's called the **principle of
indifference**: if you have no reason to prefer one hypothesis over another,
assign them equal probability.

As evidence accumulates, the prior matters less and less. If you have enough
evidence, even a wrong prior gets corrected. This is one of the most
important properties of Bayesian inference:

> *With enough evidence, all Bayesians — regardless of their prior —
> converge to the same posterior.*

Turing's team didn't need a perfect prior. They needed good evidence. And
Enigma operators, through carelessness, gave them plenty of it.


In [ ]:
# Demonstrate prior wash-out: different priors converge with enough evidence
from src.enigma_bayes.bayes.probability import bayesian_update, normalize
import numpy as np

# Three analysts with very different priors for 3 hypotheses
prior_A = np.array([0.1, 0.1, 0.8])   # strongly believes H3
prior_B = np.array([1/3, 1/3, 1/3])   # uniform (indifferent)
prior_C = np.array([0.7, 0.2, 0.1])   # strongly believes H1 (wrong!)

# Ten pieces of evidence, each strongly favouring H2
evidence = np.array([0.05, 0.90, 0.05])  # likelihood vector

for name, prior in [('Analyst A', prior_A), ('Analyst B', prior_B), ('Analyst C', prior_C)]:
    p = prior.copy()
    for _ in range(10):
        p = bayesian_update(p, evidence)
    print(f'{name}: prior={prior} → posterior after 10 updates = {p.round(4)}')

## Summary

We've derived Bayes' theorem and seen exactly how it maps onto the Enigma
codebreaking problem:

$$P(\text{setting} \mid \text{crib matches}) = \frac{P(\text{crib matches} \mid \text{setting}) \cdot P(\text{setting})}{P(\text{crib matches})}$$

Key takeaways:

1. **Prior** = uniform (all settings equally likely at the start)
2. **Likelihood** = 0 for settings that violate the no-self-encryption constraint, scored by letter frequency otherwise
3. **Posterior** = updated belief after each piece of evidence
4. **Sequential updating**: each new crib multiplies the posterior, concentrating probability on the correct setting

**In Chapter 3**, we'll build the actual Bayesian decoder and watch it crack
a message in real time — visualising the posterior collapsing onto the
correct Enigma setting.
